# M6c · Build `mart_device_monthly_telemetry`

**Goal:** roll up `fact_harsh_event` and `fact_telemetry_daily` into a
single per-(tenant, device, year_month) mart that feeds the unified ML
view (`v_ml_features_full`).

**SQL file:** `sql/25_mart_device_monthly_telemetry.sql`.
Single parameter: `:touched_months` — pass `None` to recompute every month
present in the underlying facts (full backfill).

**Why a separate mart from `mart_device_monthly_behavior`?**
Keeping the original (tested) mart untouched protects existing baseline
risk-score consumers. The new mart is opt-in and joined at view time.

**Exit criteria:**
- One row per (tenant, device, year_month) for every device with archive
  pings in that month.
- `harsh_events_per_100km` reasonable when joined to `total_distance_km`
  (typical: 0–10 events / 100 km; outliers > 30 indicate threshold mis-calibration).


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs — preview the SQL and supporting fact counts

In [ ]:
from accent_fleet.db.sql_loader import load_sql
print(load_sql('25_mart_device_monthly_telemetry.sql')[:900], '...')

import pandas as pd
with get_engine().connect() as conn:
    print('--- supporting fact counts ---')
    for t in ('fact_harsh_event', 'fact_telemetry_daily'):
        n = conn.execute(text(f'SELECT COUNT(*) FROM warehouse.{t}')).scalar_one()
        print(f'  warehouse.{t:25s}  {n:>10,}  rows')


## 3. Execute — full recompute (pass NULL touched_months)

In [ ]:
from accent_fleet.db import run_sql_file, transaction
from accent_fleet.pipeline.run_log import begin_run, end_run
import time

run_id = begin_run(mode='notebook:03_build_mart_device_monthly_telemetry')
t0 = time.time()
try:
    with transaction() as conn:
        result = run_sql_file(
            conn,
            '25_mart_device_monthly_telemetry.sql',
            params={'touched_months': None, 'etl_run_id': run_id},
        )
        rows = result.rowcount or 0
    end_run(run_id, status='success', rows_loaded=rows)
    print(f'done in {time.time()-t0:.1f}s — rows: {rows:,}')
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise


## 4. Inspect — coverage and sample rows

In [ ]:
import pandas as pd
with get_engine().connect() as conn:
    coverage = pd.read_sql(text('''
        SELECT year_month,
               COUNT(*) AS rows,
               SUM(harsh_event_total) AS harsh_total,
               SUM(total_idle_minutes)::numeric(20,1) AS idle_minutes,
               AVG(monthly_idle_ratio)::numeric(6,3)  AS avg_idle_ratio
          FROM marts.mart_device_monthly_telemetry
         GROUP BY year_month
         ORDER BY year_month DESC LIMIT 12
    '''), conn)
    sample = pd.read_sql(text('SELECT * FROM marts.mart_device_monthly_telemetry ORDER BY harsh_event_total DESC LIMIT 5'), conn)
display(coverage); display(sample)
print('OK — mart_device_monthly_telemetry built.')
